In [1]:
import pandas as pd
import numpy as np

### 1. Dropping Unnecessary Columns and Handling Missing Values
The original dataset is read in chunks to efficiently handle its large size. The `unit` column is dropped, and rows with missing `aqi` values are removed.

In [ ]:
# Define columns to be used later and initialize an empty list for chunks
pollutant_columns = ['pm2.5_avg', 'pm10_avg', 'so2_avg']
processed_chunks = []

# Read the original CSV in chunks for memory efficiency
for chunk in pd.read_csv('air_quality.csv', chunksize=50000, low_memory=False):
    # Drop the 'unit' column if it exists
    if 'unit' in chunk.columns:
        chunk = chunk.drop(columns=['unit'])

    # Remove rows where aqi is missing
    chunk = chunk.dropna(subset=['aqi'])

    # Replace placeholder '-' with numpy.nan
    existing_pollutants = [col for col in pollutant_columns if col in chunk.columns]
    for col in existing_pollutants:
        chunk[col] = chunk[col].replace('-', np.nan)

    # Convert pollutant columns to numeric, coercing errors to NaN
    if existing_pollutants:
        chunk[existing_pollutants] = chunk[existing_pollutants].apply(pd.to_numeric, errors='coerce')

    processed_chunks.append(chunk)

# Combine all processed chunks into a single DataFrame
final_df = pd.concat(processed_chunks, ignore_index=True)

# Drop rows with invalid or missing dates
final_df = final_df.dropna(subset=['date'])

# Save the cleaned data to a new CSV file
final_df.to_csv('cleaned_air_quality.csv', index=False)